# TRANSFORMERS
Este notebok muestra el código de los modelos avanzados pero se han ejecutado en el notebook de CatBoostClassifier_embeddings.ipynb

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier, Pool

import optuna
import warnings
warnings.filterwarnings("ignore")

In [ ]:
train_prep = pd.read_csv("../data/train_preprocessed.csv")
test_prep = pd.read_csv("../data/test_preprocessed.csv")

print("Train:", train_prep.shape)
print("Test:", test_prep.shape)

display(train_prep.head())
display(test_prep.head())

# CatBoost: K-Fold Ensemble

In [ ]:
N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

oof_proba = np.zeros(len(X))
test_proba_folds = np.zeros((len(X_test), N_SPLITS))

fold_scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n==============================")
    print(f"Fold {fold + 1} / {N_SPLITS}")
    print(f"==============================")
    
    X_tr = X.iloc[train_idx]
    X_val = X.iloc[valid_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]
    
    train_pool_fold = Pool(X_tr, y_tr, cat_features=cat_feature_indices)
    valid_pool_fold = Pool(X_val, y_val, cat_features=cat_feature_indices)
    test_pool_fold = Pool(X_test, cat_features=cat_feature_indices)
    
    model = CatBoostClassifier(**best_params)
    
    model.fit(
        train_pool_fold,
        eval_set=valid_pool_fold,
        early_stopping_rounds=100,
        verbose=False
    )
    
    val_proba = model.predict_proba(valid_pool_fold)[:, 1]
    test_proba_fold = model.predict_proba(test_pool_fold)[:, 1]
    
    oof_proba[valid_idx] = val_proba
    test_proba_folds[:, fold] = test_proba_fold
    
    val_pred_05 = (val_proba >= 0.5).astype(int)
    
    fold_scores.append({
        "fold": fold + 1,
        "accuracy": accuracy_score(y_val, val_pred_05),
        "f1_macro": f1_score(y_val, val_pred_05, average="macro"),
        "f1_class_1": f1_score(y_val, val_pred_05),
        "best_iteration": model.get_best_iteration()
    })
    
    models.append(model)

fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

print("F1 macro medio:", fold_scores_df["f1_macro"].mean())

In [ ]:
thresholds = np.arange(0.55, 0.72, 0.005)

threshold_results_oof = []

for threshold in thresholds:
    oof_pred = (oof_proba >= threshold).astype(int)
    
    threshold_results_oof.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y, oof_pred),
        "f1_macro": f1_score(y, oof_pred, average="macro"),
        "f1_class_1": f1_score(y, oof_pred)
    })

threshold_results_oof = pd.DataFrame(threshold_results_oof)

display(threshold_results_oof.sort_values("f1_macro", ascending=False).head(20))

best_threshold_oof = threshold_results_oof.sort_values("f1_macro", ascending=False).iloc[0]["threshold"]

print("Mejor threshold OOF:", best_threshold_oof)

In [ ]:
test_proba_ensemble = test_proba_folds.mean(axis=1)

thresholds_to_try = [
    0.625,
    0.63,
    0.635,
    0.64,
    0.645,
    0.65,
    round(float(best_threshold_oof), 3)
]

thresholds_to_try = sorted(set(thresholds_to_try))

for threshold in thresholds_to_try:
    test_predictions = (test_proba_ensemble >= threshold).astype(int)
    
    submission = pd.DataFrame({
        "id": test_prep["id"],
        "label": test_predictions
    })
    
    filename = f"submission_catboost_kfold_threshold_{str(threshold).replace('.', '')}.csv"
    submission.to_csv(filename, index=False)
    
    print("\n==============================")
    print("Archivo:", filename)
    print("Threshold:", threshold)
    print("Distribución:")
    print(submission["label"].value_counts())

# ModeloS adicionales: Transformers
Además del modelo CatBoost con embeddings MiniLM, se probaron modelos Transformer entrenados directamente para la clasificación binaria.

En este caso, el modelo recibe un texto enriquecido que combina la afirmación con metadatos contextuales como el tema, el hablante, el cargo, el estado y el partido político.

# DistilBERT

In [ ]:
import sys

!{sys.executable} -m pip install transformers datasets accelerate scikit-learn

In [ ]:
import random
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
transformer_text_cols = [
    "statement_clean",
    "subject_clean",
    "speaker_grouped_clean",
    "speaker_job_grouped_clean",
    "state_info_clean",
    "party_affiliation_clean"
]

for col in transformer_text_cols:
    train_prep[col] = train_prep[col].fillna("unknown").astype(str)
    test_prep[col] = test_prep[col].fillna("unknown").astype(str)

train_prep["transformer_text"] = (
    "Statement: " + train_prep["statement_clean"] +
    " Subject: " + train_prep["subject_clean"] +
    " Speaker: " + train_prep["speaker_grouped_clean"] +
    " Job: " + train_prep["speaker_job_grouped_clean"] +
    " State: " + train_prep["state_info_clean"] +
    " Party: " + train_prep["party_affiliation_clean"]
)

test_prep["transformer_text"] = (
    "Statement: " + test_prep["statement_clean"] +
    " Subject: " + test_prep["subject_clean"] +
    " Speaker: " + test_prep["speaker_grouped_clean"] +
    " Job: " + test_prep["speaker_job_grouped_clean"] +
    " State: " + test_prep["state_info_clean"] +
    " Party: " + test_prep["party_affiliation_clean"]
)

display(train_prep[["transformer_text", "label"]].head())

In [ ]:
def train_transformer_model(MODEL_NAME, num_epochs=2, batch_size=4, max_length=192):
    
    print("=" * 70)
    print("Entrenando modelo:", MODEL_NAME)
    print("=" * 70)
    
    train_df, valid_df = train_test_split(
        train_prep[["transformer_text", "label"]],
        test_size=0.2,
        random_state=SEED,
        stratify=train_prep["label"]
    )
    
    test_df_transformer = test_prep[["id", "transformer_text"]].copy()
    
    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))
    test_dataset = Dataset.from_pandas(test_df_transformer.reset_index(drop=True))
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    def tokenize_function(batch):
        return tokenizer(
            batch["transformer_text"],
            truncation=True,
            max_length=max_length
        )
    
    train_tokenized = train_dataset.map(tokenize_function, batched=True)
    valid_tokenized = valid_dataset.map(tokenize_function, batched=True)
    test_tokenized = test_dataset.map(tokenize_function, batched=True)
    
    train_tokenized = train_tokenized.rename_column("label", "labels")
    valid_tokenized = valid_tokenized.rename_column("label", "labels")
    
    train_tokenized = train_tokenized.remove_columns(["transformer_text"])
    valid_tokenized = valid_tokenized.remove_columns(["transformer_text"])
    test_tokenized = test_tokenized.remove_columns(["transformer_text", "id"])
    
    train_tokenized.set_format("torch")
    valid_tokenized.set_format("torch")
    test_tokenized.set_format("torch")
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
            "f1_class_1": f1_score(labels, preds)
        }
    
    safe_model_name = MODEL_NAME.replace("/", "_")
    
    training_args = TrainingArguments(
        output_dir=f"./transformer_results_{safe_model_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        seed=SEED
    )
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=valid_tokenized,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    
    print("\nResultados en validación:")
    print(eval_results)
    
    valid_outputs = trainer.predict(valid_tokenized)
    valid_logits = valid_outputs.predictions
    valid_proba = torch.softmax(torch.tensor(valid_logits), dim=1).numpy()[:, 1]
    
    print("\nResultados con threshold 0.5:")
    
    valid_pred_05 = (valid_proba >= 0.5).astype(int)
    
    print("Accuracy:", accuracy_score(valid_df["label"], valid_pred_05))
    print("F1 macro:", f1_score(valid_df["label"], valid_pred_05, average="macro"))
    print("F1 clase 1:", f1_score(valid_df["label"], valid_pred_05))
    print(classification_report(valid_df["label"], valid_pred_05))
    print(confusion_matrix(valid_df["label"], valid_pred_05))
    
    thresholds = np.arange(0.30, 0.80, 0.005)
    
    threshold_results = []
    
    for threshold in thresholds:
        valid_pred = (valid_proba >= threshold).astype(int)
        
        threshold_results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(valid_df["label"], valid_pred),
            "f1_macro": f1_score(valid_df["label"], valid_pred, average="macro"),
            "f1_class_1": f1_score(valid_df["label"], valid_pred)
        })
    
    threshold_results = pd.DataFrame(threshold_results)
    threshold_results = threshold_results.sort_values("f1_macro", ascending=False)
    
    display(threshold_results.head(15))
    
    best_threshold = threshold_results.iloc[0]["threshold"]
    best_f1_macro = threshold_results.iloc[0]["f1_macro"]
    
    print("\nMejor threshold:", best_threshold)
    print("Mejor F1 macro validación:", best_f1_macro)
    
    valid_pred_best = (valid_proba >= best_threshold).astype(int)
    
    print("\nResultados con mejor threshold:")
    print("Accuracy:", accuracy_score(valid_df["label"], valid_pred_best))
    print("F1 macro:", f1_score(valid_df["label"], valid_pred_best, average="macro"))
    print("F1 clase 1:", f1_score(valid_df["label"], valid_pred_best))
    print(classification_report(valid_df["label"], valid_pred_best))
    print(confusion_matrix(valid_df["label"], valid_pred_best))
    
    test_outputs = trainer.predict(test_tokenized)
    test_logits = test_outputs.predictions
    test_proba = torch.softmax(torch.tensor(test_logits), dim=1).numpy()[:, 1]
    
    thresholds_to_try = [
        round(float(best_threshold), 3),
        0.45,
        0.50,
        0.52,
        0.55,
        0.60,
        0.62,
        0.635,
        0.65
    ]
    
    thresholds_to_try = sorted(set(thresholds_to_try))
    
    for threshold in thresholds_to_try:
        test_predictions = (test_proba >= threshold).astype(int)
        
        submission = pd.DataFrame({
            "id": test_prep["id"],
            "label": test_predictions
        })
        
        filename = f"submission_{safe_model_name}_threshold_{str(threshold).replace('.', '')}.csv"
        submission.to_csv(filename, index=False)
        
        print("\n==============================")
        print("Archivo:", filename)
        print("Threshold:", threshold)
        print("Distribución:")
        print(submission["label"].value_counts())
    
    np.save(f"../data/test_proba_{safe_model_name}.npy", test_proba)
    np.save(f"../data/valid_proba_{safe_model_name}.npy", valid_proba)
    
    results = {
        "model_name": MODEL_NAME,
        "best_threshold": best_threshold,
        "best_f1_macro_valid": best_f1_macro,
        "eval_results": eval_results
    }
    
    return results

In [ ]:
distilbert_results = train_transformer_model(
    MODEL_NAME="distilbert-base-uncased",
    num_epochs=2,
    batch_size=4,
    max_length=192
)

distilbert_results

# ELECTRA-small

In [ ]:
electra_results = train_transformer_model(
    MODEL_NAME="google/electra-small-discriminator",
    num_epochs=2,
    batch_size=4,
    max_length=192
)

electra_results